# Model Output Examples: Base vs LoRA Fine-Tuned

Side-by-side outputs of **Qwen2.5-0.5B-Instruct** before and after LoRA
fine-tuning on the natural-language-to-SQL task.

Examples are loaded from `eval_results.json` (produced by `evaluate.py`), so this
notebook is **display-only**, so no model loading is required. The optional last cell
lets you run live inference on your own question if you want.

In [ ]:
import json
from IPython.display import display, HTML

data = json.load(open('eval_results.json'))
BASE = {r['question']: r for r in data['base']['results']}
FT   = data['fine_tuned']['results']
N    = data['fine_tuned']['n']

print(f"Loaded {N} test results.")
print(f"Base       : exec {data['base']['exec_accuracy']:.1%}  | match {data['base']['exec_match']:.1%}")
print(f"Fine-tuned : exec {data['fine_tuned']['exec_accuracy']:.1%}  | match {data['fine_tuned']['exec_match']:.1%}")

In [ ]:
def show(ft_row):
    """Render one question with base output, fine-tuned output, and gold SQL."""
    q = ft_row['question']
    b = BASE.get(q, {})
    status = ('correct' if ft_row['exec_match']
              else 'runs, wrong rows' if ft_row['executes']
              else 'failed to execute')
    def block(title, sql, bg):
        sql = (sql or '(no SQL produced)').replace('<', '&lt;')
        return (f"<div style='margin-top:6px'><div style='font-size:11px;color:#64748b;"
                f"text-transform:uppercase;letter-spacing:.04em'>{title}</div>"
                f"<pre style='background:{bg};color:#e2e8f0;padding:9px 11px;"
                f"border-radius:6px;white-space:pre-wrap;font-size:12.5px;margin:2px 0'>"
                f"{sql}</pre></div>")
    html = (f"<div style='border:1px solid #e2e8f0;border-radius:9px;padding:14px;"
            f"margin:10px 0;font-family:-apple-system,sans-serif'>"
            f"<div><span style='background:#eef2ff;color:#6366f1;font-size:11px;"
            f"padding:2px 7px;border-radius:5px;text-transform:capitalize'>"
            f"{ft_row['complexity'].replace('_',' ')}</span> "
            f"<b>{q}</b> ({status})</div>"
            + block('Base model', b.get('pred_sql'), '#3f1d1d')
            + block('Fine-tuned model', ft_row['pred_sql'], '#0f172a')
            + block('Gold (reference) SQL', ft_row['gold_sql'], '#052e16')
            + "</div>")
    display(HTML(html))

## 1. Where fine-tuning wins
Questions the **base model gets wrong** but the **fine-tuned model gets right**, one per difficulty level.

In [ ]:
seen = set()
for cx in ['easy', 'medium', 'hard', 'very_hard']:
    for r in FT:
        b = BASE.get(r['question'], {})
        if r['complexity'] == cx and r['exec_match'] and not b.get('exec_match') and cx not in seen:
            show(r)
            seen.add(cx)
            break

## 2. Fine-tuned model: runs but returns wrong rows
Valid PostgreSQL, but a missing filter / wrong column / wrong join changes the result set.

In [ ]:
wrong = [r for r in FT if r['executes'] and not r['exec_match']]
for r in wrong[:5]:
    show(r)

## 3. Fine-tuned model: fails to execute
Queries rejected by PostgreSQL: undefined aliases, bad joins, GROUP BY violations.

In [ ]:
fail = [r for r in FT if not r['executes']]
for r in fail[:5]:
    show(r)

## 4. (Optional) Live inference
Run this to generate SQL for your own question with either model. This **loads the
models** (~1 GB RAM, CPU), so skip it if you only want the saved examples above.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_PATH = './models/qwen2.5-0.5b-instruct'
LORA_PATH  = './lora_output'
with open('schema_prompt.txt') as f:
    SCHEMA = f.read()

def load(lora=False):
    tok = AutoTokenizer.from_pretrained(LORA_PATH if lora else MODEL_PATH)
    tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float32, device_map='cpu')
    if lora:
        m = PeftModel.from_pretrained(m, LORA_PATH)
    m.eval()
    return m, tok

def gen(model, tok, question):
    msgs = [
        {'role': 'system', 'content': 'You are a SQL expert. Given a database schema and a question, write the correct SQL query.'},
        {'role': 'user',    'content': f'Schema:{SCHEMA}\nQuestion: {question}'},
    ]
    t = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt')
    ids = t if isinstance(t, torch.Tensor) else t['input_ids']
    with torch.inference_mode():
        out = model.generate(ids, attention_mask=torch.ones_like(ids),
                             max_new_tokens=160, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

question = 'List the top 3 customers by total amount spent.'
ft_model, ft_tok = load(lora=True)
print('Question :', question)
print('Fine-tuned:', gen(ft_model, ft_tok, question))